# LLM file labeling on Colab GPU

Runs `model.generate_training_data` against an open model served by vLLM on the Colab GPU.

**Runtime → Change runtime type → GPU** (T4 works; L4 / A100 can serve a larger model).

Outputs (label cache, training JSONL) go to Google Drive, so a disconnected session resumes from the cache.
To run the gold-set evaluation, upload `model/tmp/eval_llm_labels.py` (gitignored) to the Drive folder `WORK` first.

## 1. Drive, code, source repos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/chitlist_labeling'   # cache + outputs live here
import os
os.makedirs(WORK, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Project code (push local changes first — this clones origin/master)
!rm -rf /content/Chained && git clone -q https://github.com/qpdbcoocdbqp/Chained.git /content/Chained
%cd /content/Chained
!mkdir -p model/tmp
# Gold-set eval script is gitignored; copy it from Drive if uploaded
!test -f {WORK}/eval_llm_labels.py && cp {WORK}/eval_llm_labels.py model/tmp/ && echo "eval script copied" || echo "no eval script in WORK — skipping eval"

In [ ]:
# Training repos, pinned to the same commits as the local source/ checkout
# (the gold set in eval_llm_labels.py refers to files at these commits)
SOURCE_REPOS = {
    'llama.cpp': ('https://github.com/ggml-org/llama.cpp.git', '07d937828636e305bc0cfe738b288f9ab05ff748'),
    'milvus':    ('https://github.com/milvus-io/milvus.git',   '6c8462cefb59b81b3332c622f80e384c3a6d6361'),
}
import subprocess
for name, (url, sha) in SOURCE_REPOS.items():
    d = f'/content/source/{name}'
    if not os.path.isdir(d):
        !git init -q {d} && git -C {d} remote add origin {url}
        !git -C {d} fetch -q --depth 1 origin {sha} && git -C {d} checkout -q FETCH_HEAD
    n_files = len(subprocess.run(['git', '-C', d, 'ls-files'], capture_output=True, text=True).stdout.splitlines())
    print(f'{name}: {n_files} files')

## 2. Start vLLM

The model is chosen by GPU memory; override `MODEL` to try another one.
`SERVED_NAME` is part of the label cache key, so labels from different models never mix.

In [ ]:
# vLLM in its own venv: installing it into Colab's Python swaps torch and leaves
# the preinstalled torchaudio on another CUDA build, which breaks the import.
# --torch-backend=auto picks the torch build matching the GPU driver.
!pip install -q uv python-dotenv pyyaml openai
!pip uninstall -y -q torchaudio
!sed -n '15,30p' /usr/local/lib/python3.12/dist-packages/transformers/loss/loss_rnnt.py
!python -c "import transformers.loss.loss_utils; print('transformers OK')"
!vllm --version
!test -x /content/vllm-env/bin/vllm || (uv venv -q --python 3.12 --seed /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm --torch-backend=auto)
!/content/vllm-env/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu ok:', torch.cuda.is_available())"

In [ ]:
# nvidia-smi rather than torch: pip just replaced torch in this session
gpu_gb = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total',
                                      '--format=csv,noheader,nounits'], text=True).split()[0]) / 1024
if gpu_gb < 20:   MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'    # T4 16GB
elif gpu_gb < 40: MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'   # L4 24GB
else:             MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'   # A100 40/80GB
SERVED_NAME = MODEL.split('/')[-1].lower()
print(f'{gpu_gb:.0f} GB GPU -> {MODEL}')

# max-model-len: prompt (~6k tokens for 20 files) + max_tokens 8192 must fit
!nohup vllm serve {MODEL} --served-model-name {SERVED_NAME} --dtype half --max-model-len 16384 --gpu-memory-utilization 0.90 --port 8000 > /content/vllm.log 2>&1 &

In [ ]:
# Wait for the server (first start downloads the weights: a few minutes)
import time, urllib.request
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5); print('vLLM ready'); break
    except Exception:
        time.sleep(10)
else:
    print('vLLM did not start — see the log below')
!tail -n 20 /content/vllm.log

In [ ]:
os.environ['LOCAL_LLM_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['LOCAL_LLM_MODEL']    = SERVED_NAME
os.environ['LOCAL_LLM_API_KEY']  = 'not-needed'

## 3. Check label quality on the gold set

Compare accuracy / precision / recall with the sonnet numbers before labeling everything.

In [ ]:
!test -f model/tmp/eval_llm_labels.py && python -m model.tmp.eval_llm_labels --source /content/source --repeat 3

## 4. Label

`--workers` keeps several batches in flight so vLLM can batch them. Re-running this cell after a disconnect
(re-run sections 1–2 first) only labels files missing from the cache.

In [ ]:
OUTPUT = f'{WORK}/llm_training_data_{SERVED_NAME}.jsonl'
!python -m model.generate_training_data --root /content/source --output {OUTPUT} --cache {WORK}/llm_label_cache.jsonl --workers 24

In [ ]:
# Replies the parser had to salvage or gave up on
!cp model/tmp/llm_failed_replies.log {WORK}/ 2>/dev/null; tail -c 2000 model/tmp/llm_failed_replies.log 2>/dev/null || echo "no failed replies"

Download `llm_training_data_<model>.jsonl` from Drive and train locally (README, Step 2):

```bash
python -m model.train_ml_scorer train --data ./model/tmp/llm_training_data_<model>.jsonl --model llm_scorer_v3.pkl --sources llm,auto
```